In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="2M0WCpQerr7nk6aJ5Zkj")
project = rf.workspace("hashimali").project("carnumberplatedetection-kdjoh")
version = project.version(1)
dataset = version.download("yolov8")


In [ ]:
# Install required packages
!pip install roboflow ultralytics opencv-python pillow torch torchvision

from roboflow import Roboflow
from ultralytics import YOLO
import cv2
import os
import shutil
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch
from pathlib import Path
import json

In [ ]:
def download_dataset():
    """Download dataset from Roboflow with proper error handling"""
    try:
        # Initialize Roboflow
        rf = Roboflow(api_key="2M0WCpQerr7nk6aJ5Zkj")

        # Access your project
        project = rf.workspace("hashimali").project("carnumberplatedetection-kdjoh")
        version = project.version(1)

        # Download dataset in YOLOv8 format
        dataset = version.download("yolov8")

        print(f"Dataset downloaded successfully to: {dataset.location}")
        return dataset

    except Exception as e:
        print(f"Error downloading dataset: {e}")
        return None

# Download the dataset
dataset = download_dataset()

In [ ]:
def validate_dataset_structure(dataset_path):
    """Validate dataset structure and fix common issues"""

    dataset_path = Path(dataset_path)

    # Check if dataset exists
    if not dataset_path.exists():
        raise FileNotFoundError(f"Dataset path {dataset_path} does not exist")

    # Expected directories
    expected_dirs = ['train', 'valid', 'test']
    expected_files = ['data.yaml']

    # Check directories
    for dir_name in expected_dirs:
        dir_path = dataset_path / dir_name
        if not dir_path.exists():
            print(f"Warning: {dir_name} directory not found, creating...")
            dir_path.mkdir(parents=True, exist_ok=True)

        # Check for images and labels subdirectories
        images_dir = dir_path / 'images'
        labels_dir = dir_path / 'labels'

        if not images_dir.exists():
            print(f"Warning: {images_dir} not found, creating...")
            images_dir.mkdir(parents=True, exist_ok=True)

        if not labels_dir.exists():
            print(f"Warning: {labels_dir} not found, creating...")
            labels_dir.mkdir(parents=True, exist_ok=True)

    # Check data.yaml
    yaml_path = dataset_path / 'data.yaml'
    if not yaml_path.exists():
        print("Error: data.yaml not found!")
        return False

    return True

# Validate dataset structure
if dataset:
    dataset_path = dataset.location
    is_valid = validate_dataset_structure(dataset_path)
    print(f"Dataset validation: {'PASSED' if is_valid else 'FAILED'}")

In [ ]:
def validate_dataset_structure(dataset_path):
    """Validate dataset structure and fix common issues"""

    dataset_path = Path(dataset_path)

    # Check if dataset exists
    if not dataset_path.exists():
        raise FileNotFoundError(f"Dataset path {dataset_path} does not exist")

    # Expected directories
    expected_dirs = ['train', 'valid', 'test']
    expected_files = ['data.yaml']

    # Check directories
    for dir_name in expected_dirs:
        dir_path = dataset_path / dir_name
        if not dir_path.exists():
            print(f"Warning: {dir_name} directory not found, creating...")
            dir_path.mkdir(parents=True, exist_ok=True)

        # Check for images and labels subdirectories
        images_dir = dir_path / 'images'
        labels_dir = dir_path / 'labels'

        if not images_dir.exists():
            print(f"Warning: {images_dir} not found, creating...")
            images_dir.mkdir(parents=True, exist_ok=True)

        if not labels_dir.exists():
            print(f"Warning: {labels_dir} not found, creating...")
            labels_dir.mkdir(parents=True, exist_ok=True)

    # Check data.yaml
    yaml_path = dataset_path / 'data.yaml'
    if not yaml_path.exists():
        print("Error: data.yaml not found!")
        return False

    return True

# Validate dataset structure
if dataset:
    dataset_path = dataset.location
    is_valid = validate_dataset_structure(dataset_path)
    print(f"Dataset validation: {'PASSED' if is_valid else 'FAILED'}")

In [ ]:
import random
import shutil

def split_data_if_empty(dataset_path, val_ratio=0.15, test_ratio=0.0):
    """
    Splits data from the 'train' directory to 'valid' and 'test' directories
    if those directories are empty.
    """
    dataset_path = Path(dataset_path)
    train_images_dir = dataset_path / 'train' / 'images'
    train_labels_dir = dataset_path / 'train' / 'labels'
    val_images_dir = dataset_path / 'valid' / 'images'
    val_labels_dir = dataset_path / 'valid' / 'labels'
    test_images_dir = dataset_path / 'test' / 'images'
    test_labels_dir = dataset_path / 'test' / 'labels'

    # Check if validation directory is empty
    if not any(val_images_dir.iterdir()):
        print("Validation set is empty. Splitting data from training set...")

        all_train_images = list(train_images_dir.glob('*.jpg')) + list(train_images_dir.glob('*.png')) # Add other image formats if necessary
        random.shuffle(all_train_images)

        num_val = int(len(all_train_images) * val_ratio)
        num_test = int(len(all_train_images) * test_ratio)

        val_files = all_train_images[:num_val]
        test_files = all_train_images[num_val : num_val + num_test]
        # The remaining_train_files are implicitly kept in train_images_dir

        # Ensure target directories exist (validate_dataset_structure already handled this, but good for self-contained function)
        val_images_dir.mkdir(parents=True, exist_ok=True)
        val_labels_dir.mkdir(parents=True, exist_ok=True)
        test_images_dir.mkdir(parents=True, exist_ok=True)
        test_labels_dir.mkdir(parents=True, exist_ok=True)

        print(f"Moving {len(val_files)} images and labels to validation set.")
        for img_path in val_files:
            label_path = train_labels_dir / (img_path.stem + '.txt')
            if img_path.exists():
                shutil.move(img_path, val_images_dir / img_path.name)
            if label_path.exists():
                shutil.move(label_path, val_labels_dir / label_path.name)

        print(f"Moving {len(test_files)} images and labels to test set.")
        for img_path in test_files:
            label_path = train_labels_dir / (img_path.stem + '.txt')
            if img_path.exists():
                shutil.move(img_path, test_images_dir / img_path.name)
            if label_path.exists():
                shutil.move(label_path, test_labels_dir / label_path.name)

        print("Data split complete.")
    else:
        print("Validation set already populated or not empty. Skipping data split.")


def train_yolov8_model(dataset_path):
    """Train YOLOv8 model with proper configuration"""

    # Load YOLOv8 model (you can choose different sizes: n, s, m, l, x)
    model = YOLO('yolov8n.pt')  # Using nano version for faster training

    # Training configuration
    training_config = {
        'data': str(Path(dataset_path) / 'data.yaml'),
        'epochs': 100,
        'imgsz': 640,
        'batch': 16,
        'patience': 10,
        'save': True,
        'device': 'cuda' if torch.cuda.is_available() else 'cpu',
        'workers': 8,
        'optimizer': 'auto',
        'lr0': 0.01,
        'lrf': 0.01,
        'momentum': 0.937,
        'weight_decay': 0.0005,
        'warmup_epochs': 3.0,
        'warmup_momentum': 0.8,
        'box': 7.5,
        'cls': 0.5,
        'dfl': 1.5,
        'name': 'license_plate_detection'
    }

    print("Starting training...")
    print(f"Using device: {training_config['device']}")

    # Start training
    results = model.train(**training_config)

    return model, results

# Train the model
if dataset:
    # First, ensure the dataset has a validation split if it's empty
    split_data_if_empty(dataset.location, val_ratio=0.15) # Using 15% of train for validation
    model, training_results = train_yolov8_model(dataset.location)

In [ ]:
# NEXT STEP: Test the trained model on user image
print("="*60)
print("STEP 3: TESTING TRAINED MODEL ON USER IMAGE")
print("="*60)

# Load the trained model (using the best weights from training)
model_path = '/content/runs/detect/license_plate_detection/weights/best.pt'
print(f"Loading trained model from: {model_path}")

from ultralytics import YOLO
model = YOLO(model_path)
print("✅ Model loaded successfully!")

# Define the test image path
test_image_path = '/content/ross-sokolovski-wLs33MTHejQ-unsplash.jpg'
print(f"Test image: {test_image_path}")

# Perform detection with optimal settings from training
print("\n🔄 Performing object detection...")
results = model.predict(
    source=test_image_path,
    conf=0.25,      # Same as training validation
    iou=0.45,       # Same as training validation
    imgsz=640,      # Same as training
    save=True,
    project='runs/detect',
    name='final_test'
)

print("✅ Detection completed!")

# Display detailed results
print("\n" + "="*50)
print("DETECTION RESULTS")
print("="*50)

for i, r in enumerate(results):
    print(f"Image: {Path(test_image_path).name}")

    if r.boxes is not None and len(r.boxes) > 0:
        print(f"Number of detections: {len(r.boxes)}")
        print("-" * 30)

        for j, box in enumerate(r.boxes):
            confidence = box.conf.item()
            class_id = int(box.cls.item())
            class_name = model.names[class_id]
            bbox = box.xyxy[0].cpu().numpy().tolist()

            print(f"Detection {j+1}:")
            print(f"  Class: {class_name} (ID: {class_id})")
            print(f"  Confidence: {confidence:.3f}")
            print(f"  Bounding Box: [x1:{bbox[0]:.1f}, y1:{bbox[1]:.1f}, x2:{bbox[2]:.1f}, y2:{bbox[3]:.1f}]")
            print()
    else:
        print("No objects detected!")
        print("Try adjusting the confidence threshold if you expect detections.")

# Display the annotated image
print("\n🖼️ DISPLAYING RESULTS")
print("="*50)

saved_image_path = Path(results[0].save_dir) / Path(test_image_path).name

if saved_image_path.exists():
    from PIL import Image
    import matplotlib.pyplot as plt

    annotated_image = Image.open(saved_image_path)

    # Create a comparison display
    original_image = Image.open(test_image_path)

    plt.figure(figsize=(16, 8))

    # Original image
    plt.subplot(1, 2, 1)
    plt.imshow(original_image)
    plt.axis('off')
    plt.title('Original Image', fontsize=14, fontweight='bold')

    # Annotated image with detections
    plt.subplot(1, 2, 2)
    plt.imshow(annotated_image)
    plt.axis('off')
    detection_count = len(results[0].boxes) if results[0].boxes is not None else 0
    plt.title(f'Detected License Plates: {detection_count} found',
              fontsize=14, fontweight='bold', color='green')

    plt.tight_layout()
    plt.show()

    print(f"✅ Annotated image saved to: {saved_image_path}")
else:
    print("❌ Annotated image not found!")

# Save detailed results to JSON
print("\n💾 SAVING DETAILED RESULTS")
print("="*50)

import json

results_data = {
    'model_info': {
        'model_path': model_path,
        'classes': model.names,
        'training_mAP50': 0.886  # From your training results
    },
    'test_image': test_image_path,
    'detection_settings': {
        'confidence_threshold': 0.25,
        'iou_threshold': 0.45,
        'image_size': 640
    },
    'detections': []
}

for r in results:
    if r.boxes is not None:
        for box in r.boxes:
            detection_info = {
                'class_id': int(box.cls.item()),
                'class_name': model.names[int(box.cls.item())],
                'confidence': float(box.conf.item()),
                'bbox': {
                    'x1': float(box.xyxy[0][0].cpu().numpy()),
                    'y1': float(box.xyxy[0][1].cpu().numpy()),
                    'x2': float(box.xyxy[0][2].cpu().numpy()),
                    'y2': float(box.xyxy[0][3].cpu().numpy())
                }
            }
            results_data['detections'].append(detection_info)

# Save to JSON
results_json_path = 'runs/detect/final_test/detection_results.json'
with open(results_json_path, 'w') as f:
    json.dump(results_data, f, indent=2)

print(f"✅ Detailed results saved to: {results_json_path}")

# Performance summary
print("\n📊 PERFORMANCE SUMMARY")
print("="*50)
print(f"Model: license_plate_detection (best.pt)")
print(f"Training mAP50: 0.886")
print(f"Test image: {Path(test_image_path).name}")
print(f"Objects detected: {len(results_data['detections'])}")
print(f"Detection confidence threshold: 0.25")

if results_data['detections']:
    avg_confidence = sum(d['confidence'] for d in results_data['detections']) / len(results_data['detections'])
    print(f"Average detection confidence: {avg_confidence:.3f}")

print("\n🎯 NEXT STEPS:")
print("1. Review the detection results above")
print("2. Check the annotated image for accuracy")
print("3. If needed, adjust confidence threshold and test again")
print("4. You can now use this model for new images!")

# Task
Test the trained model on the image `'/content/dmitry-tomashek-MgfKoRdI948-unsplash.jpg'` to perform object detection, display the annotated image, and print any detected objects.

### Execute User Image Testing

### Subtask:
Run the cell that loads the model (from `/content/runs/detect/license_plate_detection/weights/best.pt`) and calls the `test_on_single_image` function with `'/content/dmitry-tomashek-MgfKoRdI948-unsplash.jpg'`. This will perform object detection on the specified image.


**Reasoning**:
The subtask requires loading the trained YOLOv8 model, performing object detection on a specified image, and then displaying the results. This code block will perform these steps by loading the model, running predictions on the provided image, printing the detection details, and finally showing the annotated image.



In [ ]:
import os
from google.colab import files
import torch
from pathlib import Path

# Check if the model path exists
model_path = '/content/runs/detect/license_plate_detection/weights/best.pt'
if not os.path.exists(model_path):
    print("Model file not found. Please upload your model file.")

    # Upload the model file
    uploaded = files.upload()

    # Get the uploaded filename
    for filename in uploaded.keys():
        print(f"Uploaded: {filename}")
        # If it's a .pt file, use it as model
        if filename.endswith('.pt'):
            model_path = f"/content/{filename}"
            break
    else:
        print("No .pt model file found in upload!")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import tempfile
import os
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

def create_upload_interface():
    """Create an interface for uploading and testing images"""

    # Create upload button
    upload_btn = widgets.FileUpload(
        description='Upload Image',
        multiple=False,
        accept='.jpg,.jpeg,.png,.bmp'
    )

    # Create process button
    process_btn = widgets.Button(
        description='Detect License Plates',
        button_style='success',
        tooltip='Process the uploaded image'
    )

    # Create output area
    output = widgets.Output()

    def on_process_click(b):
        with output:
            clear_output()

            if not upload_btn.value:
                print("Please upload an image first!")
                return

            # Get uploaded image
            uploaded_file = list(upload_btn.value.values())[0]

            # Save to temporary file
            with tempfile.NamedTemporaryFile(delete=False, suffix='.jpg') as tmp_file:
                tmp_file.write(uploaded_file['content'])
                temp_image_path = tmp_file.name

            try:
                # Load and display original image
                original_image = Image.open(temp_image_path)

                print("Processing image...")

                # Try to load model (you'll need to adjust the path)
                try:
                    from ultralytics import YOLO

                    # Try different possible model paths
                    possible_paths = [
                        '/content/runs/detect/license_plate_detection/weights/best.pt',
                        '/content/best.pt',
                        '/content/runs/detect/train/weights/best.pt',
                        'best.pt'
                    ]

                    model = None
                    for path in possible_paths:
                        if os.path.exists(path):
                            model = YOLO(path)
                            print(f"Loaded model from: {path}")
                            break

                    if model is None:
                        print("No model file found. Please ensure your model is uploaded.")
                        # Display original image
                        plt.figure(figsize=(12, 6))
                        plt.subplot(1, 2, 1)
                        plt.imshow(original_image)
                        plt.axis('off')
                        plt.title('Original Image (No Model Available)')
                        plt.show()
                        return

                    # Perform detection
                    results = model.predict(temp_image_path, save=True, conf=0.5)

                    # Get saved image path
                    saved_image_path = Path(results[0].save_dir) / Path(temp_image_path).name

                    # Display results
                    print("\n" + "="*50)
                    print("DETECTION RESULTS")
                    print("="*50)

                    detection_count = 0
                    for r in results:
                        if r.boxes is not None:
                            detection_count = len(r.boxes)
                            for i, box in enumerate(r.boxes):
                                confidence = box.conf.item()
                                class_id = int(box.cls.item())
                                bbox = box.xyxy[0].cpu().numpy()
                                print(f"Detection {i+1}:")
                                print(f"  Class: {class_id}")
                                print(f"  Confidence: {confidence:.3f}")
                                print(f"  Bounding Box: {bbox}")

                    print(f"\nTotal detections: {detection_count}")

                    # Display side-by-side comparison
                    annotated_image = Image.open(saved_image_path)

                    plt.figure(figsize=(15, 7))

                    # Original image
                    plt.subplot(1, 2, 1)
                    plt.imshow(original_image)
                    plt.axis('off')
                    plt.title('Original Image')

                    # Annotated image
                    plt.subplot(1, 2, 2)
                    plt.imshow(annotated_image)
                    plt.axis('off')
                    plt.title(f'Detected License Plates ({detection_count} found)')

                    plt.tight_layout()
                    plt.show()

                    print(f"\nAnnotated image saved to: {saved_image_path}")

                except Exception as e:
                    print(f"Error loading model or processing image: {e}")
                    # Display just the original image
                    plt.figure(figsize=(10, 8))
                    plt.imshow(original_image)
                    plt.axis('off')
                    plt.title('Uploaded Image (Processing Failed)')
                    plt.show()

            except Exception as e:
                print(f"Error processing image: {e}")

            finally:
                # Clean up temporary file
                if os.path.exists(temp_image_path):
                    os.unlink(temp_image_path)

    process_btn.on_click(on_process_click)

    # Display interface
    print("📷 LICENSE PLATE DETECTION INTERFACE")
    print("="*40)
    print("1. Upload an image using the button below")
    print("2. Click 'Detect License Plates' to process")
    print("="*40)

    display(upload_btn)
    display(process_btn)
    display(output)

# Create the interface
create_upload_interface()

In [ ]:
import os
from pathlib import Path
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt
import json

# Check if model exists
model_path = '/content/runs/detect/license_plate_detection/weights/best.pt'

if not os.path.exists(model_path):
    print("❌ Model file not found at:", model_path)
    print("\nLet's search for existing model files or upload a new one...")

    # Search for any .pt files in the content directory
    import glob
    model_files = glob.glob('/content/**/*.pt', recursive=True)

    if model_files:
        print("📁 Found existing model files:")
        for i, model_file in enumerate(model_files):
            print(f"   {i+1}. {model_file}")

        # Use the first found model file
        model_path = model_files[0]
        print(f"\n✅ Using model: {model_path}")
    else:
        print("📤 No model files found. Please upload your model file (.pt file):")
        uploaded = files.upload()

        # Get the uploaded .pt file
        for filename in uploaded.keys():
            if filename.endswith('.pt'):
                model_path = f"/content/{filename}"
                print(f"✅ Uploaded model: {filename}")
                break
        else:
            print("❌ No .pt model file was uploaded!")
            model_path = None
else:
    print(f"✅ Model found at: {model_path}")

# Now load the model and test on the specified image
if model_path and os.path.exists(model_path):
    try:
        from ultralytics import YOLO

        print(f"\n🔄 Loading model from: {model_path}")
        model = YOLO(model_path)
        print("✅ Model loaded successfully!")

        # Define the test image path
        test_image_path = '/content/dmitry-tomashek-MgfKoRdI948-unsplash.jpg'

        # Check if test image exists
        if not os.path.exists(test_image_path):
            print(f"❌ Test image not found: {test_image_path}")
            print("\n📤 Please upload the test image:")
            uploaded_images = files.upload()
            for filename in uploaded_images.keys():
                if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                    test_image_path = f"/content/{filename}"
                    print(f"✅ Uploaded test image: {filename}")
                    break
        else:
            print(f"✅ Test image found: {test_image_path}")

        if os.path.exists(test_image_path):
            print(f"\n🎯 Testing model on: {test_image_path}")

            # Perform detection
            results = model.predict(
                source=test_image_path,
                conf=0.25,
                iou=0.45,
                imgsz=640,
                save=True,
                project='runs/detect',
                name='user_test'
            )

            # Get the saved annotated image path
            saved_image_dir = results[0].save_dir
            saved_image_path = Path(saved_image_dir) / Path(test_image_path).name

            print(f"\n📊 DETECTION RESULTS:")
            print("=" * 50)

            detection_count = 0
            for r in results:
                boxes = r.boxes
                if boxes is not None:
                    detection_count = len(boxes)
                    for i, box in enumerate(boxes):
                        confidence = box.conf.item()
                        class_id = int(box.cls.item())
                        class_name = model.names[class_id]
                        bbox = box.xyxy[0].cpu().numpy().tolist()

                        print(f"Detection {i+1}:")
                        print(f"  Class: {class_name} (ID: {class_id})")
                        print(f"  Confidence: {confidence:.3f}")
                        print(f"  Bounding Box: {bbox}")
                        print()

            print(f"Total detections: {detection_count}")

            # Display the annotated image
            if os.path.exists(saved_image_path):
                print(f"\n🖼️ Displaying annotated image from: {saved_image_path}")

                annotated_image = Image.open(saved_image_path)

                plt.figure(figsize=(15, 8))

                # Display original and annotated side by side
                original_image = Image.open(test_image_path)

                plt.subplot(1, 2, 1)
                plt.imshow(original_image)
                plt.axis('off')
                plt.title('Original Image')

                plt.subplot(1, 2, 2)
                plt.imshow(annotated_image)
                plt.axis('off')
                plt.title(f'Detected Objects ({detection_count} found)')

                plt.tight_layout()
                plt.show()

                # Save results to JSON
                results_data = {
                    'image_path': test_image_path,
                    'model_used': model_path,
                    'detections': []
                }

                for r in results:
                    boxes = r.boxes
                    if boxes is not None:
                        for box in boxes:
                            detection = {
                                'class_id': int(box.cls.item()),
                                'class_name': model.names[int(box.cls.item())],
                                'confidence': float(box.conf.item()),
                                'bbox': box.xyxy[0].cpu().numpy().tolist()
                            }
                            results_data['detections'].append(detection)

                # Save to JSON file
                results_json_path = 'runs/detect/user_test/detection_results.json'
                os.makedirs(os.path.dirname(results_json_path), exist_ok=True)
                with open(results_json_path, 'w') as f:
                    json.dump(results_data, f, indent=2)

                print(f"\n💾 Results saved to: {results_json_path}")

            else:
                print("❌ Annotated image not found!")

        else:
            print("❌ No test image available!")

    except Exception as e:
        print(f"❌ Error during model loading or inference: {e}")
        print("\nPlease check:")
        print("1. The model file is a valid YOLO model")
        print("2. All required dependencies are installed")
        print("3. The image file exists and is accessible")

else:
    print("❌ Cannot proceed without a model file!")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Image, clear_output
from PIL import Image as PImage
import io
import numpy as np
import os
from pathlib import Path

# Ensure the model and data_config_path are available from previous steps
# model = YOLO('yolov8n.pt') # Assuming 'model' is already loaded from training step
# data_config_path = 'carnumberplatedetection-1/data.yaml' # Assuming this is set

def predict_on_uploaded_image(change):
    clear_output(wait=True)
    print("Image uploaded. Performing inference...")
    for name, file_info in uploader.value.items():
        img_bytes = file_info['content']
        pil_img = PImage.open(io.BytesIO(img_bytes))
        # For YOLO's predict method, a PIL Image is often directly acceptable.

        # Perform inference
        # The 'model' variable is available from the training step
        results = model.predict(source=pil_img, save=True, conf=0.5) # Using pil_img directly

        # Display results (assuming results object contains paths to annotated images)
        if results and results[0].save_dir:
            # The YOLOv8 model saves the annotated image inside a subfolder
            # within results[0].save_dir, typically named 'predict' or 'predictN'.
            # We need to find the correct subfolder and the image within it.

            # Get the base directory where results are saved (e.g., /content/runs/detect/user_test)
            save_base_dir = Path(results[0].save_dir)

            # The original filename without path, to match the saved file name
            original_filename_stem = Path(name).stem
            original_filename_suffix = Path(name).suffix

            found_annotated_image_path = None
            # Iterate through potential 'predict' subdirectories
            for sub_dir in save_base_dir.iterdir():
                if sub_dir.is_dir() and sub_dir.name.startswith('predict'):
                    potential_path = sub_dir / f"{original_filename_stem}{original_filename_suffix}"
                    if potential_path.exists():
                        found_annotated_image_path = potential_path
                        break # Found the image, no need to search further

            if found_annotated_image_path:
                annotated_image_path = str(found_annotated_image_path)
                display(Image(filename=annotated_image_path, width=600))
            else:
                print(f"Could not find the annotated image at the expected path. Checked inside {save_base_dir}/predictN.")
                print(f"Contents of {save_base_dir}: {os.listdir(save_base_dir)}")
                print("Please check if the filename was altered or if it's in a subfolder.")
        else:
            print("No results or save directory/path found for the annotated image.")

        # Optionally print confidence scores and bounding boxes
        for r in results:
            for box in r.boxes:
                print(f"Class: {model.names[int(box.cls[0])]}, Confidence: {box.conf[0]:.2f}, Bounding Box: {box.xyxy[0].tolist()}")

# Create a file upload widget
uploader = widgets.FileUpload(accept='image/*', multiple=False)

# Set up observer to call function when file is uploaded
uploader.observe(predict_on_uploaded_image, names='value')

print("Upload an image to test the model:")
display(uploader)

In [ ]:
pip install pytesseract

## Fix apt-get Syntax

### Subtask:
Modify cell `weuwuRPz2XNJ` by adding an exclamation mark `!` before `apt-get update` and `apt-get install` so that these commands are correctly executed as shell commands.


**Reasoning**:
The subtask requires modifying an existing cell to fix a syntax error. I will use the `code_block` command to update the content of cell `weuwuRPz2XNJ` by adding exclamation marks to the `apt-get` commands, making them executable as shell commands in the notebook environment.



In [ ]:
!apt-get update
!apt-get install -y tesseract-ocr tesseract-ocr-eng
print("Tesseract OCR engine and English language pack installed.")

## Final Task

### Subtask:
Confirm that Tesseract OCR is installed without errors, which is necessary for any subsequent text recognition tasks.


## Summary:

### Q&A
The task implicitly asked to confirm that Tesseract OCR is installed without errors.
Yes, Tesseract OCR and its English language pack are confirmed to be installed, and the installation process executed without errors.

### Data Analysis Key Findings
*   The `apt-get update` command executed successfully, updating the package lists.
*   The `apt-get install` command also executed successfully, confirming that `tesseract-ocr` and `tesseract-ocr-eng` were either already installed at their newest versions or were successfully installed.
*   A confirmation message, "Tesseract OCR engine and English language pack installed," was displayed, indicating the successful completion of the installation process.
*   No syntax errors were encountered during the execution, confirming the successful fix of the `apt-get` command syntax.

### Insights or Next Steps
*   Tesseract OCR is correctly set up and ready for use in subsequent text recognition tasks.
*   The system is now prepared for any operations requiring Tesseract, such as image processing for text extraction.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, Image, clear_output
from PIL import Image as PImage
import io
import numpy as np
import pytesseract
import cv2
import os

# Ensure the model and data_config_path are available from previous steps
# model = YOLO('yolov8n.pt') # Assuming 'model' is already loaded from training step
# data_config_path = 'carnumberplatedetection-1/data.yaml' # Assuming this is set

def predict_on_uploaded_image(change):
    clear_output(wait=True)
    print("Image uploaded. Performing inference and OCR...")
    for name, file_info in uploader.value.items():
        img_bytes = file_info['content']
        pil_img = PImage.open(io.BytesIO(img_bytes))

        # Convert PIL Image to OpenCV format (numpy array)
        # OpenCV uses BGR by default, PIL uses RGB, so convert color space
        img_np = np.array(pil_img)
        img_cv2 = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)

        # Perform inference
        results = model.predict(source=img_cv2, save=True, conf=0.25)

        extracted_texts = []
        # Iterate through detections and perform OCR
        for r in results:
            for box in r.boxes:
                # Extract bounding box coordinates
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

                # Ensure coordinates are within image bounds
                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(img_cv2.shape[1], x2)
                y2 = min(img_cv2.shape[0], y2)

                # Crop the region of interest (ROI)
                roi = img_cv2[y1:y2, x1:x2]

                if roi.shape[0] == 0 or roi.shape[1] == 0:
                    continue # Skip empty ROIs

                # Convert ROI to grayscale
                gray_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)

                # Apply binary thresholding to enhance text
                _, thresh_roi = cv2.threshold(gray_roi, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

                # Perform OCR on the processed ROI
                text = pytesseract.image_to_string(thresh_roi, config='--psm 7') # --psm 7 for single line of text
                text = text.strip()

                confidence = box.conf[0]
                class_name = model.names[int(box.cls[0])]

                if text:
                    extracted_texts.append(f"Class: {class_name}, Confidence: {confidence:.2f}, BBox: [{x1}, {y1}, {x2}, {y2}], Extracted Text: '{text}'")
                else:
                    extracted_texts.append(f"Class: {class_name}, Confidence: {confidence:.2f}, BBox: [{x1}, {y1}, {x2}, {y2}], Extracted Text: 'No text found'")

        # Corrected: Display results (annotated image)
        if results and results[0].save_dir and results[0].path:
            # Construct the path to the annotated image using save_dir and the generated filename from results[0].path
            annotated_image_path = os.path.join(results[0].save_dir, os.path.basename(results[0].path))
            if os.path.exists(annotated_image_path):
                display(Image(filename=annotated_image_path, width=400))
            else:
                print(f"Could not find the annotated image at {annotated_image_path}. Listing save directory contents for debugging:")
                !ls -F {results[0].save_dir}
                print("Please check if the filename was altered or if it's in a subfolder.")
        else:
            print("No results or save directory/path found for the annotated image.")

        # Print extracted texts
        if extracted_texts:
            print("\nOCR Results:")
            for text_line in extracted_texts:
                print(text_line)
        else:
            print("No objects detected for OCR.")

# Create a file upload widget
uploader = widgets.FileUpload(accept='image/*', multiple=False)

# Set up observer to call function when file is uploaded
uploader.observe(predict_on_uploaded_image, names='value')

print("Upload an image to test the model:")
display(uploader)


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import cv2
import numpy as np
import pytesseract
from PIL import Image
import os
import tempfile
from pathlib import Path

def process_video_detection(change):
    clear_output(wait=True)
    print("Video uploaded. Processing...")

    for filename, file_info in video_uploader.value.items():
        # Save uploaded video to temporary file
        with tempfile.NamedTemporaryFile(delete=False, suffix='.mp4') as tmp_file:
            tmp_file.write(file_info['content'])
            video_path = tmp_file.name

        try:
            # Open video
            cap = cv2.VideoCapture(video_path)
            if not cap.isOpened():
                print("Error: Could not open video")
                return

            # Get video properties
            fps = int(cap.get(cv2.CAP_PROP_FPS))
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

            print(f"Video Info: {width}x{height}, {fps} FPS, {total_frames} frames")

            # Create output directory
            output_dir = 'runs/detect/video_results'
            os.makedirs(output_dir, exist_ok=True)

            # Output video path
            output_video_path = f'{output_dir}/annotated_video.mp4'
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

            frame_count = 0
            detection_results = []

            print("Processing video frames...")

            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                frame_count += 1

                # Process every 5th frame to speed up processing (adjust as needed)
                if frame_count % 5 != 0:
                    continue

                # Perform YOLO detection
                results = model.predict(source=frame, conf=0.25, iou=0.45, imgsz=640, verbose=False)

                frame_detections = []

                for r in results:
                    if r.boxes is not None:
                        for box in r.boxes:
                            # Extract bounding box coordinates
                            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                            confidence = box.conf.item()
                            class_id = int(box.cls.item())
                            class_name = model.names[class_id]

                            # Calculate object size
                            obj_width = x2 - x1
                            obj_height = y2 - y1
                            obj_area = obj_width * obj_height

                            # Draw bounding box
                            color = (0, 255, 0) if class_name.lower() in ['license_plate', 'plate'] else (255, 0, 0)
                            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

                            # Add label
                            label = f"{class_name} {confidence:.2f}"
                            cv2.putText(frame, label, (x1, y1-10),
                                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

                            # If it's a license plate, perform OCR
                            extracted_text = ""
                            if class_name.lower() in ['license_plate', 'plate']:
                                # Crop license plate region
                                roi = frame[y1:y2, x1:x2]

                                if roi.shape[0] > 0 and roi.shape[1] > 0:
                                    # Preprocess for OCR
                                    gray_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)

                                    # Apply various preprocessing techniques
                                    # 1. Gaussian blur to reduce noise
                                    blurred = cv2.GaussianBlur(gray_roi, (3, 3), 0)

                                    # 2. Different thresholding methods
                                    _, thresh_binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
                                    thresh_adaptive = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                                                           cv2.THRESH_BINARY, 11, 2)

                                    # Try OCR with different preprocessing
                                    texts = []
                                    for processed_roi, method in [(thresh_binary, 'binary'),
                                                                 (thresh_adaptive, 'adaptive')]:
                                        try:
                                            text = pytesseract.image_to_string(
                                                processed_roi,
                                                config='--psm 8 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
                                            ).strip()
                                            if text:
                                                texts.append((text, method))
                                        except:
                                            pass

                                    # Choose the best text (longest non-empty text)
                                    if texts:
                                        extracted_text = max(texts, key=lambda x: len(x[0]))[0]

                                        # Add OCR text to frame
                                        cv2.putText(frame, f"Plate: {extracted_text}",
                                                   (x1, y2+20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

                            # Store detection info
                            detection_info = {
                                'frame': frame_count,
                                'class': class_name,
                                'confidence': confidence,
                                'bbox': [x1, y1, x2, y2],
                                'size': {
                                    'width': obj_width,
                                    'height': obj_height,
                                    'area': obj_area
                                },
                                'position': {
                                    'center_x': (x1 + x2) // 2,
                                    'center_y': (y1 + y2) // 2
                                },
                                'extracted_text': extracted_text
                            }
                            frame_detections.append(detection_info)

                detection_results.extend(frame_detections)

                # Write frame to output video
                out.write(frame)

                # Progress update
                if frame_count % 50 == 0:
                    print(f"Processed {frame_count}/{total_frames} frames")

            # Release video objects
            cap.release()
            out.release()

            # Save detection results
            import json
            results_file = f'{output_dir}/detection_results.json'
            with open(results_file, 'w') as f:
                json.dump(detection_results, f, indent=2)

            # Generate summary statistics
            total_detections = len(detection_results)
            car_detections = [d for d in detection_results if d['class'].lower() in ['car', 'vehicle']]
            plate_detections = [d for d in detection_results if d['class'].lower() in ['license_plate', 'plate']]

            plates_with_text = [d for d in plate_detections if d['extracted_text']]

            print("\n" + "="*60)
            print("VIDEO PROCESSING COMPLETE")
            print("="*60)
            print(f"Total frames processed: {frame_count}")
            print(f"Total detections: {total_detections}")
            print(f"Car detections: {len(car_detections)}")
            print(f"License plate detections: {len(plate_detections)}")
            print(f"Plates with extracted text: {len(plates_with_text)}")

            if plates_with_text:
                print("\nExtracted License Plate Texts:")
                unique_plates = set(d['extracted_text'] for d in plates_with_text if d['extracted_text'])
                for plate_text in unique_plates:
                    print(f"  - {plate_text}")

            # Display the output video
            print(f"\nOutput video saved to: {output_video_path}")
            print(f"Detailed results saved to: {results_file}")

            # Show video in notebook
            display(HTML(f"""
            <video width="800" controls>
                <source src="{output_video_path}" type="video/mp4">
                Your browser does not support the video tag.
            </video>
            <p><strong>Annotated Video Output</strong></p>
            """))

            # Display sample frames with detections
            print("\nSample Detection Frames:")
            display_sample_frames(output_video_path, detection_results)

        except Exception as e:
            print(f"Error processing video: {e}")
            import traceback
            traceback.print_exc()

        finally:
            # Clean up temporary file
            if os.path.exists(video_path):
                os.unlink(video_path)

def display_sample_frames(video_path, detection_results):
    """Display sample frames with detections"""
    cap = cv2.VideoCapture(video_path)

    # Get frames with the most detections
    frame_detection_counts = {}
    for detection in detection_results:
        frame_num = detection['frame']
        frame_detection_counts[frame_num] = frame_detection_counts.get(frame_num, 0) + 1

    # Get top 3 frames with most detections
    top_frames = sorted(frame_detection_counts.items(), key=lambda x: x[1], reverse=True)[:3]

    for frame_num, count in top_frames:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
        ret, frame = cap.read()
        if ret:
            # Convert BGR to RGB for display
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # Create PIL image
            pil_img = Image.fromarray(frame_rgb)

            # Display
            print(f"Frame {frame_num}: {count} detections")
            display(pil_img.resize((600, 400)))

    cap.release()

# Create video upload widget
video_uploader = widgets.FileUpload(
    accept='video/*,.mp4,.avi,.mov,.mkv',
    multiple=False,
    description='Upload Video'
)

# Set up observer
video_uploader.observe(process_video_detection, names='value')

print("🎥 VIDEO DETECTION INTERFACE")
print("="*50)
print("Upload a video file to detect:")
print("- Cars and vehicles")
print("- License plates")
print("- Extract license plate text")
print("- Track object sizes and positions")
print("="*50)
display(video_uploader)

# Additional function for real-time camera feed (optional)
def start_camera_detection():
    """Start real-time detection from webcam"""
    print("Starting camera detection...")

    # This would require additional setup for webcam access
    # For Colab, this might need different approach
    pass

# Uncomment to add camera option
# camera_btn = widgets.Button(description="Start Camera Detection")
# camera_btn.on_click(lambda x: start_camera_detection())
# display(camera_btn)

In [ ]:
from google.colab import files

# Path to your best trained model
model_path = '/content/runs/detect/license_plate_detection/weights/best.pt'

# Check if the model exists before attempting to download
import os
if os.path.exists(model_path):
    print(f"Downloading model from: {model_path}")
    files.download(model_path)
    print("✅ Model download initiated.")
else:
    print(f"❌ Model not found at: {model_path}. Please ensure the model exists before downloading.")